# 37. Event-level reading of the altcoin backcast: market-episode clusters, effect sizes, lead times, M0 vs M1

Here I reread the notebook 17 results (10 altcoins, 2018–2019, primary definition d4) at the event level.
I use only what notebook 17 stored in `HOLDOUT17_multicoin_2018_2019_v1/`: alert times (`A|method|d4` in `phase2/{coin}.npz`), event lists (`phase2/{coin}_d4_events.csv`), regime deciles (`TT`, `DD`) and `phase2/{coin}_meta.json`, plus `COIN_MODELS_FROZEN.json`.
I do not recompute models, scores or thresholds, and the preregistered notebook 17 decisions (H1, H2, H3) stay unchanged.
After redrawing the three notebook 17 nulls (regime-matched, joint circular, joint 4h–1d) with the notebook 17 seeds, I report per-event catches and lead times, per-event null catch probabilities p_e, excess catches Σ_e (o_e − p_e) by market-episode size, an episode-level test, a market-episode cluster bootstrap with leave-one-coin-out, and paired pre-onset comparisons of M0 vs M1 and M0 vs GBM (exact McNemar).
A market episode links the breach times of the 10 coins by single linkage when neighboring breaches are at most G minutes apart (main G = 240, the outer limit of the judging window; sensitivity G = 60).
This analysis is exploratory. I write outputs only to `altcoin_event_level_v1`; they contain no prices (times, counts, probabilities and statistics only).
I record the pre-specified readings in `37_SPEC.json`; none of them is a pass/fail decision, and all numbers are descriptive.

In [1]:
from pathlib import Path
import os, json, time, hashlib
import numpy as np
import pandas as pd
from scipy.stats import binomtest
try:
    from IPython.display import display
except ImportError:
    display = print
try:
    from google.colab import drive
    drive.mount('/content/drive')
    SPOT_BASE = Path('/content/drive/MyDrive') / 'btc_lead_cost_results_77months_spot'
except ImportError:
    SPOT_BASE = Path(os.environ.get('NB37_SPOT_BASE', '.'))
    print('Not running in Colab: SPOT_BASE =', SPOT_BASE)

IN17 = SPOT_BASE / 'HOLDOUT17_multicoin_2018_2019_v1'          # read only
OUT = SPOT_BASE / 'altcoin_event_level_v1'; OUT.mkdir(parents=True, exist_ok=True)
assert OUT.resolve() != IN17.resolve()

# --- constants of notebook 17 ---
NS = 60 * 10**9
LEAD_MIN = 30; LEAD_MAX = 240
REGIME_WINDOW_D = 30
COINS = ['ETHUSDT', 'XRPUSDT', 'LTCUSDT', 'BNBUSDT', 'ADAUSDT', 'EOSUSDT', 'TRXUSDT', 'ETCUSDT', 'XLMUSDT', 'NEOUSDT']
H_START, H_END = '2018-01-01', '2020-01-01'
DEFS = {'d3': 0.03, 'd4': 0.04, 'd5': 0.05}; PRIMARY_DEF = 'd4'
METHOD_NAMES = ['M0_VF_logit', 'M1_vol_threshold', 'M2_VPIN', 'M3_GBM_rich']; PRIMARY = ['M0_VF_logit', 'M1_vol_threshold', 'M3_GBM_rich']
N_PRIMARY = 5000; N_SECONDARY = 1000; SEED = 20261031
KEYS = ['pre', 'post', 'caught', 'onset_anchored']

# --- parameters of this notebook ---
DNAME = PRIMARY_DEF; DI = list(DEFS).index(DNAME)
METHODS = PRIMARY                                                # M0, M1, GBM
NULLS = ['regime-matched', 'joint circular', 'joint 4h-1d']
GAPS = [240, 60]; GAP_MAIN = 240
SIZE_BINS = [(1, 1, '1 coin'), (2, 4, '2-4 coins'), (5, 10, '5+ coins')]
B_BOOT = 10_000; SEED37 = 20261103
SHORT = {'M0_VF_logit': 'M0', 'M1_vol_threshold': 'M1', 'M3_GBM_rich': 'GBM'}
T0 = time.time()


def ts(s):
    return pd.Timestamp(s, tz='UTC')


def sha256_file(p):
    return hashlib.sha256(Path(p).read_bytes()).hexdigest()


# --- from notebook 17 (match_events, onset_anchored, counts) ---
def match_events(events, alerts):
    alerts = np.sort(np.asarray(alerts, dtype=np.int64))
    out = []; used = set()
    for ev in events.itertuples(index=False):
        breach = ev.breach_utc.value; onset = ev.onset_1pct_utc.value
        left = int(np.searchsorted(alerts, breach - LEAD_MAX * NS, 'left'))
        right = int(np.searchsorted(alerts, breach - LEAD_MIN * NS, 'right'))
        for j in range(left, right):
            if j not in used:
                used.add(j)
                out.append(dict(event_id=int(ev.event_id), breach_utc=ev.breach_utc,
                                alarm_utc=pd.Timestamp(int(alerts[j]), tz='UTC'),
                                lead_min=(breach - int(alerts[j])) / NS,
                                pre1pct=bool(alerts[j] < onset)))
                break
    return pd.DataFrame(out, columns=['event_id', 'breach_utc', 'alarm_utc', 'lead_min', 'pre1pct'])


def onset_anchored(ev, a):
    used = np.zeros(len(a), bool); n = 0
    for o in np.sort(ev.onset_1pct_utc.astype('int64').to_numpy()):
        l_ = np.searchsorted(a, o - LEAD_MAX * NS, 'left'); r_ = np.searchsorted(a, o, 'left')
        for j in range(l_, r_):
            if not used[j]:
                used[j] = True; n += 1; break
    return n


def counts(ev, a):
    a = np.sort(np.asarray(a, np.int64))
    if len(ev) == 0:
        return dict(pre=0, post=0, caught=0, onset_anchored=0)
    m = match_events(ev, a); pre = int(m.pre1pct.sum()) if len(m) else 0
    return dict(pre=pre, post=len(m) - pre, caught=len(m), onset_anchored=onset_anchored(ev, a))

Mounted at /content/drive


## 0. Notebook 17 files

In [2]:
lock = IN17 / 'HOLDOUT17_EVALUATED.lock'
if not lock.is_file():
    raise RuntimeError(f'Notebook 17 lock file not found: {lock}. Point to a folder where the notebook 17 evaluation has finished.')
LOCK17 = json.loads(lock.read_text())
FROZEN_SHA = sha256_file(IN17 / 'COIN_MODELS_FROZEN.json')
if FROZEN_SHA != LOCK17['frozen_sha256']:
    raise RuntimeError('GUARD FAIL: COIN_MODELS_FROZEN.json differs from the hash in the notebook-17 lock')
print('notebook 17 lock:', LOCK17['evaluated_utc'], '| frozen models sha256 ok')

# DATA as in notebook 17 (same construction and order, so the random draws are the same)
DATA = {}
for sym in COINS:
    meta = json.loads((IN17 / 'phase2' / f'{sym}_meta.json').read_text())
    if meta.get('excluded'):
        continue
    if meta['frozen_sha256'] != FROZEN_SHA:
        raise RuntimeError(f'GUARD FAIL: {sym} phase-2 files were made with different frozen models')
    z = np.load(IN17 / 'phase2' / f'{sym}.npz')
    evs = {}
    for dname in DEFS:
        p = IN17 / 'phase2' / f'{sym}_{dname}_events.csv'
        if p.is_file():
            e = pd.read_csv(p, usecols=['event_id', 'peak_utc', 'onset_1pct_utc', 'breach_utc', 'confirmed_utc',
                                        'peak_to_breach_minutes', 'onset_to_breach_minutes'])     # no price columns
            for c_ in ['peak_utc', 'onset_1pct_utc', 'breach_utc', 'confirmed_utc']:
                e[c_] = pd.to_datetime(e[c_], utc=True)
            evs[dname] = e
    t0 = ts(meta['eval_start']).value
    TT = z['TT']; DD = z['DD'].astype(int); order = np.lexsort((TT, DD))
    DATA[sym] = dict(meta=meta, ev=evs, A={k[2:]: z[k] for k in z.files if k.startswith('A|')}, t0=t0, L=(ts(H_END).value - t0) // NS,
                     TT=TT, DD=DD, P=TT[order], DS=DD[order],
                     starts=np.searchsorted(DD[order], np.arange(10), 'left'), ends=np.searchsorted(DD[order], np.arange(10), 'right'))
USED = list(DATA); LMIN = min(d['L'] for d in DATA.values())
print('coins:', USED)

PC17 = pd.read_csv(IN17 / 'H17_per_coin_counts.csv'); R17 = pd.read_csv(IN17 / 'H17_all_tests.csv')
bad = []
for s, d in DATA.items():
    for m in METHOD_NAMES:
        a = d['A'][f'{m}|{DNAME}']
        if len(a) != d['meta']['alerts'][f'{m}|{DNAME}']:
            bad.append((s, m, 'alerts', len(a), d['meta']['alerts'][f'{m}|{DNAME}']))
        c = counts(d['ev'][DNAME], a)
        r = PC17[(PC17.coin == s) & (PC17.method == m) & (PC17.definition == DNAME)].iloc[0]
        for k in KEYS:
            if c[k] != int(r[k]):
                bad.append((s, m, k, c[k], int(r[k])))
    b = d['ev'][DNAME].breach_utc.astype('int64').to_numpy()
    if len(b) > 1 and np.diff(np.sort(b)).min() <= (LEAD_MAX - LEAD_MIN) * NS:
        bad.append((s, 'judging windows overlap'))
if bad:
    raise RuntimeError(f'GUARD FAIL (per-coin counts): {bad[:10]}')
print('guard 1 ok: per-coin alert counts and pre/post/caught/onset-anchored counts equal notebook 17;'
      ' judging windows of different events do not overlap')

# Global event index (all coins, d4) and fast matcher.
# Judging windows do not overlap, so the greedy match of notebook 17 gives each event the first alert in its window;
# guard 2 checks this against match_events on every observed alert set.
EV = []
for s, d in DATA.items():
    e = d['ev'][DNAME].copy(); e['coin'] = s; EV.append(e)
EV = pd.concat(EV, ignore_index=True)
EV['gidx'] = np.arange(len(EV))
SL = {s: np.flatnonzero(EV.coin.to_numpy() == s) for s in USED}
BR = {s: EV.breach_utc.astype('int64').to_numpy()[SL[s]] for s in USED}
ON = {s: EV.onset_1pct_utc.astype('int64').to_numpy()[SL[s]] for s in USED}


def fast_match(s, a):
    """Per event of coin s: first alert in [b - 240, b - 30] (or -1) and whether it is before the 1% onset."""
    a = np.sort(np.asarray(a, np.int64)); b = BR[s]
    lo = np.searchsorted(a, b - LEAD_MAX * NS, 'left'); hi = np.searchsorted(a, b - LEAD_MIN * NS, 'right')
    hit = hi > lo
    al = np.where(hit, a[np.minimum(lo, max(len(a) - 1, 0))] if len(a) else 0, -1)
    pre = hit & (al < ON[s])
    return hit, pre, al


bad = []
for s, d in DATA.items():
    for m in METHOD_NAMES:
        a = d['A'][f'{m}|{DNAME}']
        mm = match_events(d['ev'][DNAME], a)
        hit, pre, al = fast_match(s, a)
        ids = d['ev'][DNAME].event_id.to_numpy()
        if set(mm.event_id) != set(ids[hit]) or set(mm.event_id[mm.pre1pct]) != set(ids[pre]):
            bad.append((s, m))
        elif len(mm):
            lead_fast = dict(zip(ids[hit], (BR[s][hit] - al[hit]) / NS))
            if not np.allclose(mm.lead_min.to_numpy(), [lead_fast[i] for i in mm.event_id]):
                bad.append((s, m, 'lead'))
if bad:
    raise RuntimeError(f'GUARD FAIL (fast matcher): {bad}')
print('guard 2 ok: the fast matcher equals match_events on all observed alert sets')

OBS = {}
for m in METHODS:
    pre = np.zeros(len(EV), bool); hit = np.zeros(len(EV), bool); alert = np.full(len(EV), -1, np.int64)
    for s in USED:
        h, p, al = fast_match(s, DATA[s]['A'][f'{m}|{DNAME}'])
        hit[SL[s]] = h; pre[SL[s]] = p; alert[SL[s]] = al
    OBS[m] = dict(pre=pre, post=hit & ~pre, caught=hit, alert=alert,
                  alerts=sum(len(DATA[s]['A'][f'{m}|{DNAME}']) for s in USED))
    for k in ['pre', 'post', 'caught']:
        r = R17[(R17.method == m) & (R17.definition == DNAME) & (R17['null'] == 'regime-matched') & (R17.endpoint == k)].iloc[0]
        if int(OBS[m][k].sum()) != int(r.observed) or OBS[m]['alerts'] != int(r.alerts):
            raise RuntimeError(f'GUARD FAIL: pooled {m} {k} {int(OBS[m][k].sum())} vs notebook 17 {int(r.observed)}')
print('guard 3 ok: pooled observed counts and alert totals equal H17_all_tests.csv')

notebook 17 lock: 2026-10-29T18:03:23.023774+00:00 | frozen models sha256 ok
coins: ['ETHUSDT', 'XRPUSDT', 'LTCUSDT', 'BNBUSDT', 'ADAUSDT', 'EOSUSDT', 'TRXUSDT', 'ETCUSDT', 'XLMUSDT', 'NEOUSDT']
guard 1 ok: per-coin alert counts and pre/post/caught/onset-anchored counts equal notebook 17; judging windows of different events do not overlap
guard 2 ok: the fast matcher equals match_events on all observed alert sets
guard 3 ok: pooled observed counts and alert totals equal H17_all_tests.csv


### 0b. The three nulls with the notebook 17 seeds and per-event catches

In [3]:
# Market episodes (labels only): single linkage of breach times across coins
order_b = np.argsort(EV.breach_utc.astype('int64').to_numpy(), kind='stable')
for G in GAPS:
    bt = EV.breach_utc.astype('int64').to_numpy()[order_b]
    new = np.r_[True, np.diff(bt) > G * NS]
    epi = np.empty(len(EV), np.int64); epi[order_b] = np.cumsum(new) - 1
    EV[f'episode_{G}'] = epi
    ncoins = EV.groupby(f'episode_{G}').coin.nunique()
    EV[f'episode_{G}_coins'] = EV[f'episode_{G}'].map(ncoins).to_numpy()
EPI = EV[f'episode_{GAP_MAIN}'].to_numpy(); N_EPI = int(EPI.max()) + 1
print({G: int(EV[f'episode_{G}'].max()) + 1 for G in GAPS}, 'market episodes for gaps', GAPS)


def draws_regime(m, n, seed):
    """regime_null of notebook 17, recording per-event pre/post hits of every draw."""
    rng = np.random.default_rng(seed); W = REGIME_WINDOW_D * 1440 * NS; prep = {}
    for s, d in DATA.items():
        a = d['A'].get(f'{m}|{DNAME}')
        if a is None or len(a) == 0:
            continue
        j = np.clip(np.searchsorted(d['TT'], a, 'right') - 1, 0, len(d['TT']) - 1); q = d['DD'][j]
        lo = np.array([d['starts'][x] + np.searchsorted(d['P'][d['starts'][x]:d['ends'][x]], ti - W, 'left') for ti, x in zip(a, q)])
        hi = np.array([d['starts'][x] + np.searchsorted(d['P'][d['starts'][x]:d['ends'][x]], ti + W, 'right') for ti, x in zip(a, q)])
        prep[s] = (a, lo, hi)
    for i in range(n):
        sh = {}
        for s, (a, lo, hi) in prep.items():
            P = DATA[s]['P']; c = P[lo + (rng.random(len(a)) * (hi - lo)).astype(np.int64)]; bad = np.abs(c - a) < LEAD_MAX * NS
            for _ in range(20):
                if not bad.any():
                    break
                c[bad] = P[lo[bad] + (rng.random(int(bad.sum())) * (hi[bad] - lo[bad])).astype(np.int64)]; bad = np.abs(c - a) < LEAD_MAX * NS
            sh[s] = np.sort(c)
        yield sh


def draws_joint(m, n, seed, band=None):
    """joint_null of notebook 17 (circular shift of every coin by the same delta)."""
    rng = np.random.default_rng(seed)
    if band is None:
        deltas = rng.integers(7 * 1440, LMIN - 7 * 1440, size=n)
    else:
        deltas = rng.integers(band[0], band[1] + 1, size=n) * (rng.integers(0, 2, size=n) * 2 - 1)
    for dl in deltas:
        yield {s: np.sort(d['t0'] + (d['A'][f'{m}|{DNAME}'] - d['t0'] + int(dl) * NS) % (d['L'] * NS))
               for s, d in DATA.items() if f'{m}|{DNAME}' in d['A']}


def run_null(m, nname):
    ci = METHOD_NAMES.index(m); base = SEED + 100 * ci + 10 * DI
    n_main = N_PRIMARY if (m in PRIMARY and DNAME == PRIMARY_DEF) else N_SECONDARY
    if nname == 'regime-matched':
        gen, n = draws_regime(m, n_main, base + 1), n_main
    elif nname == 'joint circular':
        gen, n = draws_joint(m, n_main, base + 2), n_main
    else:
        gen, n = draws_joint(m, N_SECONDARY, base + 3, band=(240, 1440)), N_SECONDARY
    tot = {k: np.empty(n, np.int64) for k in ['pre', 'post', 'caught']}
    ep_hits = np.empty(n, np.int64)
    p_pre = np.zeros(len(EV)); p_post = np.zeros(len(EV))
    for i, sh in enumerate(gen):
        pre = np.zeros(len(EV), bool); hit = np.zeros(len(EV), bool)
        for s in USED:
            h, p, _ = fast_match(s, sh.get(s, DATA[s]['A'][f'{m}|{DNAME}']))
            hit[SL[s]] = h; pre[SL[s]] = p
        p_pre += pre; p_post += hit & ~pre
        tot['pre'][i] = pre.sum(); tot['post'][i] = (hit & ~pre).sum(); tot['caught'][i] = hit.sum()
        ep_hits[i] = np.unique(EPI[pre]).size
    return dict(n=n, tot=tot, ep_hits=ep_hits, p_pre=p_pre / n, p_post=p_post / n)


NULL = {}
for m in METHODS:
    for nname in NULLS:
        t_ = time.time(); NULL[(m, nname)] = run_null(m, nname)
        print(f'{SHORT[m]:>3} {nname:<15} {NULL[(m, nname)]["n"]:>5} draws ({time.time() - t_:.0f}s)')

bad = []
for (m, nname), x in NULL.items():
    for k in ['pre', 'post', 'caught']:
        r = R17[(R17.method == m) & (R17.definition == DNAME) & (R17['null'] == nname) & (R17.endpoint == k)].iloc[0]
        v = x['tot'][k]; obs = int(OBS[m][k].sum())
        mine = (float(v.mean()), float(np.percentile(v, 5)), float(np.percentile(v, 95)), (1 + int((v >= obs).sum())) / (len(v) + 1), len(v))
        ref = (float(r.null_mean), float(r.null_q05), float(r.null_q95), float(r.p_one_sided), int(r.draws))
        if not (np.allclose(mine[:4], ref[:4], rtol=0, atol=1e-9) and mine[4] == ref[4]):
            bad.append((SHORT[m], nname, k, mine, ref))
if bad:
    raise RuntimeError(f'GUARD FAIL (null reproduction): {bad[:6]}')
print('guard 4 ok: all three nulls reproduce notebook 17 (mean, 5th and 95th percentiles, p, draws) for M0, M1 and GBM')

{240: 419, 60: 548} market episodes for gaps [240, 60]
 M0 regime-matched   5000 draws (6s)
 M0 joint circular   5000 draws (4s)
 M0 joint 4h-1d      1000 draws (1s)
 M1 regime-matched   5000 draws (6s)
 M1 joint circular   5000 draws (3s)
 M1 joint 4h-1d      1000 draws (1s)
GBM regime-matched   5000 draws (8s)
GBM joint circular   5000 draws (3s)
GBM joint 4h-1d      1000 draws (1s)
guard 4 ok: all three nulls reproduce notebook 17 (mean, 5th and 95th percentiles, p, draws) for M0, M1 and GBM


## 1–2. Observed per-event catches and null catch probabilities

In [4]:
ev_out = EV[['coin', 'event_id', 'peak_utc', 'onset_1pct_utc', 'breach_utc', 'onset_to_breach_minutes',
             'episode_240', 'episode_240_coins', 'episode_60', 'episode_60_coins']].copy()
for m in METHODS:
    o = OBS[m]; k = SHORT[m]
    ev_out[f'{k}_pre'] = o['pre'].astype(int); ev_out[f'{k}_post'] = o['post'].astype(int)
    au = pd.Series(pd.to_datetime(np.clip(o['alert'], 0, None), utc=True)); au[~o['caught']] = pd.NaT
    ev_out[f'{k}_alert_utc'] = au.to_numpy()
    ev_out[f'{k}_lead_to_breach_min'] = np.where(o['caught'], (EV.breach_utc.astype('int64').to_numpy() - o['alert']) / NS, np.nan)
    ev_out[f'{k}_alert_minus_onset_min'] = np.where(o['caught'], (o['alert'] - EV.onset_1pct_utc.astype('int64').to_numpy()) / NS, np.nan)
ev_out.to_csv(OUT / '1_event_level.csv', index=False)

pr = EV[['coin', 'event_id', 'episode_240', 'episode_240_coins']].copy()
for (m, nname), x in NULL.items():
    tag = f'{SHORT[m]}|{nname}'
    pr[f'p_pre|{tag}'] = x['p_pre']; pr[f'p_post|{tag}'] = x['p_post']
pr.to_csv(OUT / '2_event_null_probability.csv', index=False)
print('events:', len(EV), '| written 1_event_level.csv, 2_event_null_probability.csv')

events: 943 | written 1_event_level.csv, 2_event_null_probability.csv


## 3–4. Excess catches by market-episode size and episode-level test

In [5]:
rows = []
for G in GAPS:
    size = EV[f'episode_{G}_coins'].to_numpy()
    for m in METHODS:
        for nname in NULLS:
            x = NULL[(m, nname)]
            for ep in ['pre', 'post']:
                o = OBS[m][ep].astype(float); p = x['p_pre'] if ep == 'pre' else x['p_post']
                for lo_, hi_, lab in SIZE_BINS + [(1, 10, 'all')]:
                    sel = (size >= lo_) & (size <= hi_)
                    rows.append(dict(gap_minutes=G, method=SHORT[m], null=nname, endpoint=ep, episode_size=lab,
                                     events=int(sel.sum()), observed=int(o[sel].sum()), expected=float(p[sel].sum()),
                                     excess=float((o - p)[sel].sum()), ratio=float(o[sel].sum() / p[sel].sum()) if p[sel].sum() > 0 else np.nan,
                                     excess_per_100_alerts=float(100 * (o - p)[sel].sum() / OBS[m]['alerts']),
                                     share_of_total_excess=float((o - p)[sel].sum() / (o - p).sum()) if (o - p).sum() != 0 else np.nan))
T3 = pd.DataFrame(rows); T3.to_csv(OUT / '3_excess_by_episode_size.csv', index=False)
print('Excess pre-onset catches by market-episode size (G = 240, regime-matched null):')
display(T3[(T3.gap_minutes == GAP_MAIN) & (T3['null'] == 'regime-matched') & (T3.endpoint == 'pre')].round(3))

rows = []
for m in METHODS:
    obs_ep = int(np.unique(EPI[OBS[m]['pre']]).size)
    for nname in NULLS:
        v = NULL[(m, nname)]['ep_hits']
        rows.append(dict(method=SHORT[m], null=nname, draws=len(v), episodes=N_EPI, observed_episodes_with_pre=obs_ep,
                         null_mean=float(v.mean()), null_q05=float(np.percentile(v, 5)), null_q95=float(np.percentile(v, 95)),
                         excess_episodes=float(obs_ep - v.mean()), ratio=float(obs_ep / v.mean()),
                         p_one_sided=(1 + int((v >= obs_ep).sum())) / (len(v) + 1)))
T4 = pd.DataFrame(rows); T4.to_csv(OUT / '4_episode_level_test.csv', index=False)
print(f'Market episodes (G = {GAP_MAIN}) with at least one pre-onset catch:'); display(T4.round(4))

Excess pre-onset catches by market-episode size (G = 240, regime-matched null):


,gap_minutes,method,null,endpoint,episode_size,events,observed,expected,excess,ratio,excess_per_100_alerts,share_of_total_excess
0,240,M0,regime-matched,pre,1 coin,222,20,10.448,9.552,1.914,0.233,0.383
1,240,M0,regime-matched,pre,2-4 coins,386,22,15.524,6.476,1.417,0.158,0.260
2,240,M0,regime-matched,pre,5+ coins,335,25,16.107,8.893,1.552,0.217,0.357
3,240,M0,regime-matched,pre,all,943,67,42.079,24.921,1.592,0.608,1.000
24,240,M1,regime-matched,pre,1 coin,222,22,10.384,11.616,2.119,0.286,0.400
25,240,M1,regime-matched,pre,2-4 coins,386,24,15.087,8.913,1.591,0.219,0.307
26,240,M1,regime-matched,pre,5+ coins,335,25,16.457,8.543,1.519,0.210,0.294
27,240,M1,regime-matched,pre,all,943,71,41.928,29.072,1.693,0.715,1.000
48,240,GBM,regime-matched,pre,1 coin,222,14,7.348,6.652,1.905,0.204,2.662
49,240,GBM,regime-matched,pre,2-4 coins,386,7,10.814,-3.814,0.647,-0.117,-1.526


Market episodes (G = 240) with at least one pre-onset catch:


,method,null,draws,episodes,observed_episodes_with_pre,null_mean,null_q05,null_q95,excess_episodes,ratio,p_one_sided
0,M0,regime-matched,5000,419,46,38.1764,29.00,47.0,7.8236,1.2049,0.0932
1,M0,joint circular,5000,419,46,34.2444,25.00,43.0,11.7556,1.3433,0.0258
2,M0,joint 4h-1d,1000,419,46,40.7370,32.00,50.0,5.2630,1.1292,0.1848
3,M1,regime-matched,5000,419,50,38.0350,29.00,47.0,11.9650,1.3146,0.0198
4,M1,joint circular,5000,419,50,34.0068,25.00,43.0,15.9932,1.4703,0.0054
5,M1,joint 4h-1d,1000,419,50,40.6880,31.00,50.0,9.3120,1.2289,0.0799
6,GBM,regime-matched,5000,419,31,29.9422,22.00,38.0,1.0578,1.0353,0.4481
7,GBM,joint circular,5000,419,31,29.6252,22.00,38.0,1.3748,1.0464,0.4225
8,GBM,joint 4h-1d,1000,419,31,25.0330,19.95,32.0,5.9670,1.2384,0.1039


## 5. Market-episode cluster bootstrap and leave-one-coin-out

In [6]:
rng = np.random.default_rng(SEED37)
IDX = rng.integers(0, N_EPI, size=(B_BOOT, N_EPI))                    # the same resampled episodes for every row
rows = []
for m in METHODS:
    for nname in NULLS:
        x = NULL[(m, nname)]
        for ep in ['pre', 'post']:
            o = OBS[m][ep].astype(float); p = x['p_pre'] if ep == 'pre' else x['p_post']
            ex_ep = np.bincount(EPI, weights=o - p, minlength=N_EPI)
            bt = 100 * ex_ep[IDX].sum(axis=1) / OBS[m]['alerts']
            pt = 100 * ex_ep.sum() / OBS[m]['alerts']
            rows.append(dict(method=SHORT[m], null=nname, endpoint=ep, alerts=OBS[m]['alerts'], episodes=N_EPI, draws=B_BOOT,
                             excess_per_100_alerts=pt, lo95=float(np.percentile(bt, 2.5)), hi95=float(np.percentile(bt, 97.5)),
                             share_draws_positive=float((bt > 0).mean()),
                             alerts_per_extra_event=float(OBS[m]['alerts'] / ex_ep.sum()) if ex_ep.sum() > 0 else np.nan))
T5 = pd.DataFrame(rows); T5.to_csv(OUT / '5_cluster_bootstrap.csv', index=False)
print(f'Excess per 100 alerts, 95% market-episode bootstrap interval (B = {B_BOOT}):'); display(T5.round(3))

rows = []
coin_arr = EV.coin.to_numpy()
for m in METHODS:
    for nname in NULLS:
        x = NULL[(m, nname)]; o = OBS[m]['pre'].astype(float)
        for s in USED:
            keep = coin_arr != s
            al = OBS[m]['alerts'] - len(DATA[s]['A'][f'{m}|{DNAME}'])
            rows.append(dict(method=SHORT[m], null=nname, coin_left_out=s, alerts=al,
                             excess_per_100_alerts=float(100 * (o - x['p_pre'])[keep].sum() / al)))
T5b = pd.DataFrame(rows); T5b.to_csv(OUT / '5b_leave_one_coin_out.csv', index=False)
display(T5b.groupby(['method', 'null']).excess_per_100_alerts.agg(['min', 'max']).round(3))

Excess per 100 alerts, 95% market-episode bootstrap interval (B = 10000):


,method,null,endpoint,alerts,episodes,draws,excess_per_100_alerts,lo95,hi95,share_draws_positive,alerts_per_extra_event
0,M0,regime-matched,pre,4101,419,10000,0.608,0.115,1.200,0.994,164.561
1,M0,regime-matched,post,4101,419,10000,2.226,1.538,2.973,1.000,44.931
2,M0,joint circular,pre,4101,419,10000,0.633,0.110,1.250,0.992,157.971
3,M0,joint circular,post,4101,419,10000,2.215,1.517,2.969,1.000,45.137
4,M0,joint 4h-1d,pre,4101,419,10000,0.241,-0.194,0.774,0.847,414.619
5,M0,joint 4h-1d,post,4101,419,10000,1.936,1.336,2.585,1.000,51.664
6,M1,regime-matched,pre,4065,419,10000,0.715,0.216,1.309,0.998,139.826
7,M1,regime-matched,post,4065,419,10000,2.336,1.614,3.122,1.000,42.799
8,M1,joint circular,pre,4065,419,10000,0.745,0.223,1.364,0.998,134.262
9,M1,joint circular,post,4065,419,10000,2.343,1.608,3.143,1.000,42.683


min    max
method null                        
GBM    joint 4h-1d     0.015  0.341
       joint circular -0.104  0.241
       regime-matched -0.084  0.228
M0     joint 4h-1d     0.127  0.359
       joint circular  0.541  0.727
       regime-matched  0.512  0.697
M1     joint 4h-1d     0.232  0.431
       joint circular  0.631  0.821
       regime-matched  0.611  0.786

## 6. Lead times

In [7]:
rows = []
for m in METHODS:
    k = SHORT[m]
    for ep in ['pre', 'post']:
        sel = OBS[m][ep]
        lb = ev_out.loc[sel, f'{k}_lead_to_breach_min'].to_numpy(); ao = ev_out.loc[sel, f'{k}_alert_minus_onset_min'].to_numpy()
        row = dict(method=k, endpoint=ep, n=int(sel.sum()))
        for nm, v in [('lead_to_breach_min', lb), ('alert_minus_onset_min', ao)]:
            if len(v):
                q = np.percentile(v, [10, 25, 50, 75, 90])
                row.update({f'{nm}_p{p_}': float(qq) for p_, qq in zip([10, 25, 50, 75, 90], q)})
        rows.append(row)
T6 = pd.DataFrame(rows); T6.to_csv(OUT / '6_lead_times.csv', index=False)
print('Lead times (minutes; alert_minus_onset < 0 means before the 1% onset):'); display(T6.round(1))

Lead times (minutes; alert_minus_onset < 0 means before the 1% onset):


,method,endpoint,n,lead_to_breach_min_p10,lead_to_breach_min_p25,lead_to_breach_min_p50,lead_to_breach_min_p75,lead_to_breach_min_p90,alert_minus_onset_min_p10,alert_minus_onset_min_p25,alert_minus_onset_min_p50,alert_minus_onset_min_p75,alert_minus_onset_min_p90
0,M0,pre,67,68.2,109.5,166.0,202.5,220.4,-147.6,-96.0,-67.0,-22.5,-7.6
1,M0,post,139,41.8,58.0,114.0,160.5,194.4,3.0,10.5,34.0,84.0,124.2
2,M1,pre,71,58.0,106.0,158.0,202.0,217.0,-137.0,-90.5,-66.0,-26.0,-7.0
3,M1,post,143,41.4,63.0,117.0,162.0,195.8,5.0,14.0,34.0,82.0,122.6
4,GBM,pre,35,91.6,123.5,163.0,205.0,214.2,-126.4,-76.5,-55.0,-23.5,-10.8
5,GBM,post,71,38.0,70.5,113.0,153.0,184.0,7.0,16.0,40.0,82.5,115.0


## 7. Events both systems catch before onset: M0 vs M1, M0 vs GBM

In [8]:
rows = []
for a_, b_ in [('M0_VF_logit', 'M1_vol_threshold'), ('M0_VF_logit', 'M3_GBM_rich')]:
    A_, B_ = OBS[a_]['pre'], OBS[b_]['pre']
    only_a, only_b = int((A_ & ~B_).sum()), int((~A_ & B_).sum())
    disc = (A_ ^ B_)
    rows.append(dict(a=SHORT[a_], b=SHORT[b_], both=int((A_ & B_).sum()), only_a=only_a, only_b=only_b, neither=int((~A_ & ~B_).sum()),
                     a_total=int(A_.sum()), b_total=int(B_.sum()),
                     mcnemar_exact_p_two_sided=float(binomtest(only_a, only_a + only_b, 0.5).pvalue) if only_a + only_b else 1.0,
                     discordant_episodes_240=int(np.unique(EPI[disc]).size),
                     same_alert_minute_when_both=int(((OBS[a_]['alert'] == OBS[b_]['alert']) & A_ & B_).sum())))
T7 = pd.DataFrame(rows); T7.to_csv(OUT / '7_paired_pre_onset.csv', index=False)
display(T7)

,a,b,both,only_a,only_b,neither,a_total,b_total,mcnemar_exact_p_two_sided,discordant_episodes_240,same_alert_minute_when_both
0,M0,M1,61,6,10,866,67,71,0.454498,15,36
1,M0,GBM,14,53,21,855,67,35,0.000256,59,1


## 8. Summary

In [9]:
SPEC = dict(notebook='37_Altcoin_Event_Level_SPOT', status='exploratory', reads=str(IN17), writes=str(OUT),
            notebook17_lock=LOCK17, frozen_models_sha256=FROZEN_SHA, definition=DNAME, methods=METHODS, nulls=NULLS,
            null_seeds={f'{SHORT[m]}|{n_}': SEED + 100 * METHOD_NAMES.index(m) + 10 * DI + 1 + NULLS.index(n_) for m in METHODS for n_ in NULLS},
            episode_gaps_minutes=GAPS, episode_gap_main=GAP_MAIN, size_bins=[b[2] for b in SIZE_BINS],
            bootstrap=dict(unit=f'market episode (G = {GAP_MAIN})', draws=B_BOOT, seed=SEED37, fixed=['p_e', 'alerts']),
            reading='descriptive; no decision rule; notebook-17 decisions unchanged; outputs carry no prices',
            guards=['per-coin alert and catch counts = H17_per_coin_counts.csv', 'fast matcher = match_events on observed alerts',
                    'pooled observed = H17_all_tests.csv', 'three nulls reproduce H17_all_tests.csv (mean, q05, q95, p, draws)'])
(OUT / '37_SPEC.json').write_text(json.dumps(SPEC, indent=2, default=str))

g = T3[(T3.gap_minutes == GAP_MAIN) & (T3.endpoint == 'pre')]
SUMMARY = dict(
    events=int(len(EV)), market_episodes={G: int(EV[f'episode_{G}'].max()) + 1 for G in GAPS},
    events_in_multi_coin_episodes={G: int((EV[f'episode_{G}_coins'] >= 2).sum()) for G in GAPS},
    excess_by_size_regime=g[g['null'] == 'regime-matched'][['method', 'episode_size', 'events', 'observed', 'expected', 'excess', 'share_of_total_excess']].to_dict('records'),
    episode_level=T4.to_dict('records'), cluster_bootstrap=T5.to_dict('records'),
    leave_one_coin_out=T5b.groupby(['method', 'null']).excess_per_100_alerts.agg(['min', 'max']).reset_index().to_dict('records'),
    lead_times=T6.to_dict('records'), paired=T7.to_dict('records'), runtime_seconds=round(time.time() - T0, 1))
(OUT / '37_SUMMARY.json').write_text(json.dumps(SUMMARY, indent=2, default=lambda o: o.item() if hasattr(o, 'item') else str(o)))
print(f'done in {time.time() - T0:.0f}s ->', OUT)

done in 63s -> /content/drive/MyDrive/btc_lead_cost_results_77months_spot/altcoin_event_level_v1
